# Phase 1 — Data Cleaning & Quality Check  

**StreamFlix — Content Analytics Project**

# Objective:
Access the quality of datasets by checking 
1. Null values
2. Duplicate rows
3. Data types
4. Referential Integrity

# 1.Load all csv files

In [ ]:
import pandas as pd
from pathlib import Path

# Load CSV files
folder = Path(r"D:\ff5a102e4361911d42cef9e21cda23d0")

csv_files = list(folder.glob("*.csv"))

dataframes = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

display(csv_files)
display(dataframes.keys()) 

for name, df in dataframes.items():
    display(f"\n--- {name} ---")
    display(df.head())

# 2. Basic Data Profiling

In [ ]:
# 2. For each table: row count, column count, data types
for name, df in dataframes.items():
    display(f"\nProfile for {name}:")
    display(f"Row count: {df.shape[0]}")
    display(f"Column count: {df.shape[1]}")
    display(f"Data types:\n{df.dtypes}")

# Missing values Analysis

In [ ]:
# 3. Missing values in every column —columns have nulls and % of rows affected.
for name, df in dataframes.items():
    display(f"\nMissing values for {name}:")
    missing_values = df.isnull().sum()
    total_rows = df.shape[0]
    for column, count in missing_values.items():
        if count > 0:
            percentage = (count / total_rows) * 100
            display(f"Column '{column}' has {count} missing values ({percentage:.2f}% of rows).")

# Duplicate Value Analysis

In [ ]:
# 4. For each table, check for duplicate rows and report the count of duplicates.
for name, df in dataframes.items():
    display(f"\nDuplicate rows for {name}:")
    duplicate_rows = df.duplicated().sum()
    display(f"Number of duplicate rows: {duplicate_rows}")

# Duplicate watch_id in watch_history.csv
if 'watch_history' in dataframes:
    watch_history_df = dataframes['watch_history']

    duplicate_watch_ids = watch_history_df[watch_history_df.duplicated(subset='watch_id', keep=False)].sort_values('watch_id')

    if duplicate_watch_ids.empty:
        display("No duplicate watch_id values found.")
    else:
        display("Duplicate watch_id values found:")
        display(duplicate_watch_ids)
else:
    display("watch_history was not found in dataframes.")
    
# Duplicate subscriber_id in subscriber.csv
if 'subscribers' in dataframes:
    subscriber_df = dataframes['subscribers']
    
    duplicate_subscriber_ids = subscriber_df[subscriber_df.duplicated(subset='subscriber_id', keep=False)].sort_values('subscriber_id') 
   
    if duplicate_subscriber_ids.empty:
        display("No duplicate subscriber_id values found.")
    else:
        display(f"\nDuplicate subscriber_id value found:")
        display(duplicate_subscriber_ids)
else:
    display("subscribers was not found in dataframes.")


# Data Type Validation

In [ ]:
# 5. Validate the data types of each column in every table and convert date columns to datetime format. 
for name, df in dataframes.items():
    display(f"\nData types for {name}:")
    display(df.dtypes)

    # Convert date columns to datetime format
    for column in df.columns:
        if 'date' in column.lower():
            try:
                df[column] = pd.to_datetime(df[column], errors='coerce')
                display(f"Converted column '{column}' to datetime format.")
            except Exception as e:
                display(f"Could not convert column '{column}' to datetime format: {e}")
    display("\nUpdated data types after conversion:")
    display(df.dtypes)

# Outlier Checking

In [ ]:
# 6. check if watch_duration_min is longer than the content and flag them.
for name, df in dataframes.items():
    if 'watch_duration_min' in df.columns and 'content_duration_min' in df.columns:
        outliers = df[df['watch_duration_min'] > df['content_duration_min']]
        if not outliers.empty:
            display(f"\nOutliers in {name} where watch_duration_min is longer than content_duration_min:")
            display(outliers)
        else:
            display(f"\nNo outliers found in {name} where watch_duration_min is longer than content_duration_min.")

# Refrential Integrity Check

In [ ]:
# 7. do all subscriber_id in watch_history.csv exist in subscribers.csv
if 'watch_history' in dataframes and 'subscribers' in dataframes:
    watch_history_df = dataframes['watch_history']
    subscriber_df = dataframes['subscribers']

    missing_subscriber_ids = watch_history_df[~watch_history_df['subscriber_id'].isin(subscriber_df['subscriber_id'])]
    if not missing_subscriber_ids.empty:
        display(f"\nsubscriber_id in watch_history that do not exist in subscribers:")
        display(missing_subscriber_ids)
    else:
        display("\nAll subscriber_id in watch_history exist in subscribers.")
        
# do all title_id in watch_history.csv exist in titles.csv
if 'watch_history' in dataframes and 'titles' in dataframes:
    watch_history_df = dataframes['watch_history']
    titles_df = dataframes['titles']

    missing_title_ids = watch_history_df[~watch_history_df['title_id'].isin(titles_df['title_id'])]
    if not missing_title_ids.empty:
        display(f"\ntitle_id in watch_history that do not exist in titles:")
        display(missing_title_ids)
    else:
        display("\nAll title_id in watch_history exist in titles.")

# Data Cleaning and Quality Report

**StreamFlix — Content Analytics Project**

# Objective:
The objective of this data quality assessment was to evaluate the accuracy, consistency and integrity of the dataset before performing any business analysis. The assessment was conducted on all six CSV files using python(pandas).

# Data profiling:
All six datasets were loaded successfully. The row count, column count and data types were reviewed. df.shape() was used to find rows and column count. Data types of every column for each data sets were examined to understand the overall structure of the data sets. The data types uses along whole data sets are:
1. str
2. int64
3. float64
4. bool

# Missing Values:
A missing value analysis was performed on every datasets. Columns containing null values were identified. There were only two data set having columns with null values.
1. In the subscribers dataset "churn_date" has null values with 74.66% rows effect.
2. In the titles dataset "liscence_expiry" has null values with 21.84% rows effect.

# Duplicate Records:
Duplicate row was checked across all datasets. Special attention was given to watch_history and subscribers table. There was no duplicate values found across all datasets.

# Data Type Validation:
The data types of all columns were checked to ensure consistency. Columns containing dates were validated as date format as they were in str(object) format.

# Outlier Detection:
Outlier analysis was performed on the watch_duration_min column to identify if it is greater than content. There were no outliers found as every row of watch_duration_min is smaller than content.

# Referential Integrity:
Relationship between tables were identified. All subscriber_id from watch_history table is exist in subscribers table and also all title_id from watch_history table is exist in title table. There were null unmatched items.

# Conclusion:
The data cleaning and quality assessment provided a comprehensive understanding of the dataset's condition before analysis. The identified issues including missing values, duplicate rows, wrong data type, potential outlier and referential integrity checks, were documented to ensure that the data sets is reliable for further analysis and dashboard creation.

# Store Cleaned dataset into file

In [ ]:
# store the cleaned dataframes back to CSV files
folder_path = r"D:\Internship project2"

for name, df in dataframes.items():
    output_file = Path(folder_path) / f"{name}_cleaned.csv"
    df.to_csv(output_file, index=False)
    display(f"Cleaned dataframe '{name}' saved to {output_file}")

# Specific checks to run

In [ ]:
# 1. Check if churn_date is after signup_date in subscribers.csv. Flag rows where it is not.
if 'subscribers' in dataframes:
    subscriber_df = dataframes['subscribers']
    
    invalid_churn_dates = subscriber_df[subscriber_df['churn_date'] < subscriber_df['signup_date']]
    if not invalid_churn_dates.empty:
        display(f"\nRows in subscribers where churn_date is before signup_date:")
        display(invalid_churn_dates)
    else:
        display("\nAll churn_date are after signup_date in subscribers.")
        
# 2. Check if actives subscriber(is_active = true) have a blank churn_date in subscribers.csv. Flag rows where it is not.
if 'subscribers' in dataframes:
    subscriber_df = dataframes['subscribers']
    
    invalid_active_subscribers = subscriber_df[(subscriber_df['is_active'] == True) & (subscriber_df['churn_date'].notnull())]
    if not invalid_active_subscribers.empty:
        display(f"\nActive subscribers with a non-blank churn_date:")
        display(invalid_active_subscribers)
    else:
        display("\nAll active subscribers have a blank churn_date.")
        
# 3. Check to confirm completion_pct roughly equals watch_duration_min ÷ content_duration_min × 100 in watch_history.csv. 
if 'watch_history' in dataframes:
    watch_history_df = dataframes['watch_history']
    
    # Calculate the expected completion percentage
    watch_history_df['expected_completion_pct'] = (watch_history_df['watch_duration_min'] / watch_history_df['content_duration_min']) * 100
    
    # Flag rows where the actual completion_pct does not roughly equal the expected completion percentage
    invalid_completion_pct = watch_history_df[abs(watch_history_df['completion_pct'] - watch_history_df['expected_completion_pct']) > 5]  # Allowing a tolerance of 5%
    
    if not invalid_completion_pct.empty:
        display(f"\nRows in watch_history where completion_pct does not roughly equal watch_duration_min ÷ content_duration_min * 100:")
        display(invalid_completion_pct)
    else:
        display("\nAll completion_pct values roughly equal watch_duration_min ÷ content_duration_min * 100.")
        
# 4. Check to Confirm sentiment only contains Positive / Neutral / Negative in reviews.csv. Flag anything else.
if 'reviews' in dataframes:
    reviews_df = dataframes['reviews']
    
    invalid_sentiments = reviews_df[~reviews_df['sentiment'].isin(['Positive', 'Neutral', 'Negative'])]
    if not invalid_sentiments.empty:
        display(f"\nRows in reviews where sentiment is not Positive, Neutral, or Negative:")
        display(invalid_sentiments)
    else:
        display("\nAll sentiment values in reviews are either Positive, Neutral, or Negative.")
        
# 5. To check every watchlist entry link to a real subscriber and a real title in watchlist.csv.
if 'watchlist' in dataframes and 'subscribers' in dataframes and 'titles' in dataframes:
    watchlist_df = dataframes['watchlist']
    subscriber_df = dataframes['subscribers']
    titles_df = dataframes['titles']
    
    # Check for invalid subscriber_id
    invalid_subscriber_ids = watchlist_df[~watchlist_df['subscriber_id'].isin(subscriber_df['subscriber_id'])]
    
    # Check for invalid title_id
    invalid_title_ids = watchlist_df[~watchlist_df['title_id'].isin(titles_df['title_id'])]
    
    if not invalid_subscriber_ids.empty:
        display(f"\nRows in watchlist where subscriber_id does not exist in subscribers:")
        display(invalid_subscriber_ids)
    else:
        display("\nAll subscriber_id in watchlist exist in subscribers.")
        
    if not invalid_title_ids.empty:
        display(f"\nRows in watchlist where title_id does not exist in titles:")
        display(invalid_title_ids)
    else:
        display("\nAll title_id in watchlist exist in titles.") 
        
# 6. Check for subscribers with very short tenure but marked active in subscribers.csv. 
if 'subscribers' in dataframes:
    subscriber_df = dataframes['subscribers']
    
    # Calculate tenure in days
    subscriber_df['tenure_days'] = (pd.to_datetime('today') - subscriber_df['signup_date']).dt.days
    
    # Flag active subscribers with tenure less than 30 days
    short_tenure_active_subscribers = subscriber_df[(subscriber_df['is_active'] == True) & (subscriber_df['tenure_days'] < 30)]
    
    if not short_tenure_active_subscribers.empty:
        display(f"\nActive subscribers with very short tenure (less than 30 days):")
        display(short_tenure_active_subscribers)
    else:
        display("\nNo active subscribers with very short tenure found.")